# W5｜長照數據處理 – pandas
**智慧醫療與機器學習基礎**　臺北醫學大學 智慧醫療學士學位學程　授課教師：王三源　課程助教：古珉瑄（醫資所，m610115009@tmu.edu.tw）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/FIHML/blob/main/notebooks/W05_pandas.ipynb)

### 今天的情境
你是 A、B、C 三間長照機構共同聘請的**資料分析助理**。主任從照護系統匯出兩個檔案給你：

| 檔案 | 內容 |
|---|---|
| `residents_raw.csv` | 住民基本資料（每人一列） |
| `assessments_raw.csv` | 每半年一次的評估紀錄（每人 1–4 筆） |

主任的問題：**「失智和沒有失智的住民，在各項評估上有什麼不同？」**

> ⚠️ 本資料為**教學用模擬資料**，姓名與數值皆為程式隨機生成，不對應任何真實個人。

### 執行環境
本教材已在 **Python 3.12＋pandas 2.2.3** 與 **pandas 3.0.6** 兩個版本測試通過。Colab 的預設版本會隨時間更新，執行下面第一格會印出你目前的版本。
- pandas 2.x：文字欄位的型別顯示為 `object`
- pandas 3.x：文字欄位的型別顯示為 `str`，意思相同，不影響本教材的程式

### 使用方式
1. 「檔案 → 在雲端硬碟中儲存複本」，之後都在自己的複本上作業。
2. 每個程式格按 **Shift + Enter** 執行，**由上往下依序執行**。
3. 看到 **▶ 實作** 的地方，請在 `# TODO` 處寫程式。

In [ ]:
import pandas as pd
import numpy as np

BASE = "https://raw.githubusercontent.com/wang3yuan/FIHML/main/data/ltc/"
import sys
print("Python 版本：", sys.version.split()[0])
print("pandas 版本：", pd.__version__)

---
# 第 1 堂｜認識你的資料
## 1.1 讀取資料
`pd.read_csv()` 可以讀本機檔案，也可以直接讀網址。

In [ ]:
res = pd.read_csv(BASE + "residents_raw.csv")
ass = pd.read_csv(BASE + "assessments_raw.csv")
res.head()

## 1.2 第一眼：`shape`、`info()`、`describe()`
拿到資料的第一件事，**不是**急著分析，而是先檢查：有幾列？幾欄？每欄是什麼型別？缺多少？

In [ ]:
print("住民資料：", res.shape)
print("評估紀錄：", ass.shape)

In [ ]:
res.info()

🔍 **觀察**：`教育年數` 應該是數字，為什麼型別不是 `int64` 或 `float64`？`體重` 的 non-null 筆數為什麼比較少？

In [ ]:
ass.describe().round(1)

🔍 **觀察**：`SPMSQ錯誤題數` 滿分是 10 題，為什麼最大值是 99？`夜間睡眠時數` 為什麼沒有出現在 `describe()` 裡？

這些「怪地方」就是第 2 堂要處理的髒資料。先記下來！

## 1.3 把欄位名稱改成英文
中文欄名可以用，但打字容易出錯，也不方便之後的程式套件處理。用一個 dict 對照表一次改好：

In [ ]:
res_cols = {
    "住民編號": "resident_id", "姓名": "name", "性別": "sex", "出生日期": "birth_date",
    "機構": "facility", "入住日期": "admission_date", "教育年數": "education_years",
    "身高": "height_cm", "體重": "weight_kg", "血型": "blood_type", "房號": "room",
    "高血壓": "hypertension", "糖尿病": "diabetes", "中風史": "stroke",
    "聽力障礙": "hearing_loss", "失智診斷": "dementia",
}
ass_cols = {
    "住民編號": "resident_id", "評估日期": "assess_date", "SPMSQ錯誤題數": "spmsq_errors",
    "巴氏量表": "adl_barthel", "IADL": "iadl", "GDS15": "gds15",
    "過去一年跌倒次數": "falls_past_year", "夜間睡眠時數": "sleep_hours",
    "每週社交活動次數": "social_per_week", "用藥品項數": "n_medications",
}
res = res.rename(columns=res_cols)
ass = ass.rename(columns=ass_cols)
res.columns.tolist()

## 1.4 選取欄位與列
| 寫法 | 意思 |
|---|---|
| `df["col"]` | 取一欄（得到 Series） |
| `df[["a", "b"]]` | 取多欄（得到 DataFrame，注意兩層中括號） |
| `df.loc[列標籤, 欄名]` | 用**名稱**選 |
| `df.iloc[列位置, 欄位置]` | 用**位置**（第幾個）選 |

In [ ]:
res["facility"].head()

In [ ]:
res[["resident_id", "sex", "facility"]].head()

In [ ]:
# 第 0–2 列、resident_id 到 facility 這幾欄
res.loc[0:2, "resident_id":"facility"]

In [ ]:
# 前 3 列、前 4 欄（iloc 不含結尾）
res.iloc[0:3, 0:4]

## 1.5 條件篩選
條件會產生一串 True/False（布林遮罩），放進 `df[...]` 就只留下 True 的列。

⚠️ 多個條件要用 `&`（且）、`|`（或），**每個條件都要加括號**。

In [ ]:
mask = res["facility"] == "A機構"
mask.head()

In [ ]:
res[mask].shape

In [ ]:
# A 機構且有中風史
res[(res["facility"] == "A機構") & (res["stroke"] == "有")].head()

In [ ]:
# isin：符合清單中任一個
res[res["blood_type"].isin(["A", "AB"])].shape

## 1.6 排序與計數

In [ ]:
ass.sort_values("adl_barthel").head()

In [ ]:
res["sex"].value_counts(dropna=False)

🔍 **觀察**：性別怎麼會有 6 種寫法？

---
## ▶ 實作 5-1（約 15 分鐘）
請用 `res` 與 `ass` 回答下列問題。

**(a)** A、B、C 三間機構各有幾位住民？（用 `value_counts()`）

In [ ]:
# TODO：請在這裡寫程式


**(b)** 列出 B 機構、有聽力障礙（`hearing_loss == "有"`）的住民，只顯示 `resident_id`、`name`、`facility` 三欄。共有幾位？

In [ ]:
# TODO：請在這裡寫程式


**(c)** 評估紀錄中，SPMSQ 錯誤題數 ≥ 5（中度以上認知障礙）的有幾筆？
再算一次「≥ 5 而且 ≤ 10」的筆數。兩個答案為什麼不一樣？

In [ ]:
# TODO：請在這裡寫程式


**(d)** 巴氏量表分數最低的 5 筆評估是哪些住民？（`sort_values` + `head`）

In [ ]:
# TODO：請在這裡寫程式


**(e) 討論**：到目前為止，你發現這兩份資料有哪些「怪地方」？至少列出 3 個。

> 你的答案：
> 1.
> 2.
> 3. 

---
# 第 2 堂｜資料清理
### 清理檢查清單
1. 重複列　2. 型別錯誤　3. 編碼不一致　4. 單位不一致　5. 缺值代碼　6. 日期格式　7. 不合理的值　8. 缺值盤點

我們先一起清理 `res`（住民基本資料），實作 5-2 再請你清理 `ass`（評估紀錄）。
清理前先留一份原始資料，方便對照：

In [ ]:
res_raw = res.copy()

## 2.1 重複列

In [ ]:
print("完全重複的列：", res.duplicated().sum())
print("重複的住民編號：", res["resident_id"].duplicated().sum())

In [ ]:
res = res.drop_duplicates()
print(res.shape, "住民編號是否唯一：", res["resident_id"].is_unique)

## 2.2 型別錯誤：文字混進數字欄
`pd.to_numeric(..., errors="coerce")`：轉不成數字的（例如「不詳」）變成 `NaN`。

In [ ]:
res["education_years"].unique()

In [ ]:
res["education_years"] = pd.to_numeric(res["education_years"], errors="coerce")
res["education_years"].describe()

## 2.3 編碼不一致：用 `map` 統一
`map` 對照表裡沒有的值會變成 `NaN`，所以要先用 `unique()` 看清楚有哪些寫法。

In [ ]:
res["sex"] = res["sex"].map({"男": "M", "M": "M", "male": "M",
                             "女": "F", "F": "F", "female": "F"})
res["sex"].value_counts(dropna=False)

In [ ]:
yes_no = {"有": 1, "Y": 1, "無": 0, "N": 0}
for col in ["hypertension", "diabetes", "stroke", "hearing_loss", "dementia"]:
    res[col] = res[col].map(yes_no)
res[["hypertension", "diabetes", "stroke", "hearing_loss", "dementia"]].mean().round(3)

## 2.4 單位不一致：身高寫成公尺
身高小於 3 的，一定是用「公尺」記錄 → 乘以 100。

In [ ]:
print("用公尺記錄的筆數：", (res["height_cm"] < 3).sum())
res.loc[res["height_cm"] < 3, "height_cm"] = res["height_cm"] * 100
res["height_cm"].describe().round(1)

## 2.5 不可能的值 → 設為缺值
體重 0 公斤不可能，是「沒量」被記成 0。

In [ ]:
print("體重為 0：", (res["weight_kg"] == 0).sum())
res.loc[res["weight_kg"] == 0, "weight_kg"] = np.nan

## 2.6 日期：西元、斜線、民國年混在一起
`pd.to_datetime(..., format="mixed")` 能處理大部分西元格式，但看不懂「民國」，要自己寫函式轉換。

In [ ]:
res["birth_date"].sample(8, random_state=3)

In [ ]:
import re

def roc_to_iso(s):
    """把「民國32年4月24日」轉成「1943-04-24」，其他格式原樣回傳。"""
    if isinstance(s, str) and s.startswith("民國"):
        y, m, d = re.findall(r"\d+", s)
        return f"{int(y) + 1911}-{int(m):02d}-{int(d):02d}"
    return s

print(roc_to_iso("民國32年4月24日"))
res["birth_date"] = pd.to_datetime(res["birth_date"].apply(roc_to_iso), format="mixed", errors="coerce")
res["admission_date"] = pd.to_datetime(res["admission_date"])
res["birth_date"].isna().sum()

## 2.7 計算年齡，並檢查不合理的值
以 2026-10-01 為基準日計算年齡。長照住民的年齡應該介於 60–110 歲。

In [ ]:
ref = pd.Timestamp("2026-10-01")
res["age"] = ((ref - res["birth_date"]).dt.days / 365.25).round().astype("Int64")
res[(res["age"] < 60) | (res["age"] > 110)][["resident_id", "birth_date", "age"]]

這幾筆應該是年份打錯（1826？2031？）。真實工作中要**回頭向機構確認**；這裡先設為缺值。

In [ ]:
bad = (res["age"] < 60) | (res["age"] > 110)
res.loc[bad, ["birth_date", "age"]] = [pd.NaT, pd.NA]
res["age"].describe().round(1)

## 2.8 缺值盤點
缺值不一定要補。先問：**為什麼會缺？** 缺值本身可能就是資訊（例如住民無法配合施測）。

In [ ]:
res.isna().sum()

In [ ]:
# 常見的補值方式（示範用，暫不覆蓋原資料）
print("教育年數中位數：", res["education_years"].median())
res["education_years"].fillna(res["education_years"].median()).isna().sum()

## 2.9 清理前 vs 清理後

In [ ]:
res_raw.info(memory_usage=False)
print("-" * 50)
res.info(memory_usage=False)

---
## ▶ 實作 5-2（約 20 分鐘）：清理評估紀錄 `ass`
依照清理檢查清單完成下列步驟，每一步都印出結果確認。

**(a)** 刪除完全重複的列。刪除前後各有幾列？

In [ ]:
# TODO：請在這裡寫程式


**(b)** `spmsq_errors` 的 99 代表「未評估」，把它換成 `NaN`。
提示：`df.loc[條件, "欄"] = np.nan` 或 `df["欄"].replace(99, np.nan)`

In [ ]:
# TODO：請在這裡寫程式


**(c)** `sleep_hours` 裡有「7.5小時」這種寫法。先把「小時」去掉，再轉成數字。
提示：`.str.replace("小時", "")`、`pd.to_numeric()`

In [ ]:
# TODO：請在這裡寫程式


**(d)** 把 `assess_date` 轉成日期型別。

In [ ]:
# TODO：請在這裡寫程式


**(e)** 檢查各量表是否都在合理範圍內：SPMSQ 0–10、巴氏量表 0–100、IADL 0–8、GDS-15 0–15。
提示：`ass[["spmsq_errors", "adl_barthel", "iadl", "gds15"]].describe()`

In [ ]:
# TODO：請在這裡寫程式


**(f)** 盤點每一欄的缺值數。哪一欄缺最多？你認為照護現場可能是什麼原因造成的？

In [ ]:
# TODO：請在這裡寫程式


> 你的想法：

---
# 第 3 堂｜合併、轉換與摘要
## 3.1 一人多筆：取每位住民最近一次評估

In [ ]:
ass["resident_id"].value_counts().value_counts().sort_index()

In [ ]:
latest = (ass.sort_values(["resident_id", "assess_date"])
              .drop_duplicates("resident_id", keep="last"))
print(len(latest), "位住民")
latest.head()

## 3.2 合併兩張表：`merge`
`validate="one_to_one"` 會幫你檢查兩邊的 key 是否都唯一，避免合併後資料悄悄變多。

In [ ]:
df = res.merge(latest, on="resident_id", how="left", validate="one_to_one")
print(res.shape, latest.shape, "→", df.shape)

## 3.3 新增欄位

In [ ]:
df["bmi"] = (df["weight_kg"] / (df["height_cm"] / 100) ** 2).round(1)
df["polypharmacy"] = (df["n_medications"] >= 5).astype(int)   # 多重用藥：≥ 5 種
df[["bmi", "polypharmacy"]].describe().round(2)

## 3.4 連續數值分組：`pd.cut`
巴氏量表依賴程度：0–20 完全依賴、21–60 嚴重依賴、61–90 中度依賴、91–99 輕度依賴、100 完全獨立。

In [ ]:
df["age_group"] = pd.cut(df["age"].astype(float), bins=[59, 74, 84, 120],
                         labels=["65–74", "75–84", "85+"])
df["adl_level"] = pd.cut(df["adl_barthel"], bins=[-1, 20, 60, 90, 99, 100],
                         labels=["完全依賴", "嚴重依賴", "中度依賴", "輕度依賴", "完全獨立"])
df["adl_level"].value_counts().sort_index()

## 3.5 分組摘要：`groupby`
split → apply → combine：依失智與否分成兩組，各自算平均，再組合成一張表。

In [ ]:
features = ["age", "education_years", "spmsq_errors", "adl_barthel", "iadl",
            "gds15", "sleep_hours", "social_per_week", "falls_past_year"]
df.groupby("dementia")[features].mean().round(2).T

In [ ]:
# 一次算多個統計量
df.groupby("facility").agg(
    住民數=("resident_id", "count"),
    失智比例=("dementia", "mean"),
    平均年齡=("age", "mean"),
).round(2)

## 3.6 交叉表：`crosstab`

In [ ]:
pd.crosstab(df["hearing_loss"], df["dementia"], normalize="index").round(3)

🔍 有聽力障礙的住民失智比例較高。這代表聽力障礙**造成**失智嗎？還是有其他可能？（提示：年齡）

## 3.7 去識別化，然後存檔
要交給別人分析之前，先移除可以直接認出個人的欄位。

In [ ]:
deid = df.drop(columns=["name", "birth_date", "room"])
deid.to_csv("ltc_clean_w05.csv", index=False)
deid.shape

在 Colab 左側「📁 檔案」可以看到並下載 `ltc_clean_w05.csv`。

---
## ▶ 實作 5-3（約 15 分鐘）

**(a)** 依 `age_group` 分組，計算每組的住民數與失智比例。

In [ ]:
# TODO：請在這裡寫程式


**(b)** 做 `adl_level` × `dementia` 的交叉表（每一列加總為 1）。依賴程度越高，失智比例如何變化？

In [ ]:
# TODO：請在這裡寫程式


**(c)** 比較有、無多重用藥（`polypharmacy`）的住民，平均 `falls_past_year` 是多少？

In [ ]:
# TODO：請在這裡寫程式


**(d)** 根據 3.5 的比較表與上面的結果，用 2–3 句話寫下你給主任的「第一個發現」。

> 你的發現：

---
## ▶ 連結你的專題：資料字典（約 10 分鐘）
期中報告要說明「**資料需求**」。請從**課程 Miro 白板**（連結請見 Im@TMU）上各組的議題中選一個（不限自己小組），或自己繼續發想一個新的議題，列出 5 個以上需要的欄位。可以和旁邊的同學討論。

議題名稱與來源：

| 欄位名稱 | 意義 | 型別／單位 | 誰記錄？多久記一次？ | 是否為個資？ | 確診前可取得？ |
|---|---|---|---|---|---|
| 範例：sleep_hours | 夜間睡眠時數 | 數值／小時 | 夜班照服員，每天 | 否 | 是 |
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |

⚠️ **若要用來預測「是否失智」：確診之後才產生的資料（失智症用藥、CDR 分期、失智專區照護計畫……）都不能當作模型的輸入**，否則等於偷看答案。

想一想：這些資料現在**有沒有人在記**？記在紙本、Excel 還是系統裡？哪一項**最難取得**？

👉 這份草稿就是作業第 5 題的起點，回家可以繼續完成。

---
### 本週 pandas 指令總整理
| 目的 | 指令 |
|---|---|
| 讀檔 | `pd.read_csv()` |
| 看結構 | `shape`、`head()`、`info()`、`describe()` |
| 改欄名 | `rename(columns={...})` |
| 選取 | `df["a"]`、`df[["a","b"]]`、`loc`、`iloc` |
| 篩選 | `df[(條件1) & (條件2)]`、`isin()` |
| 重複 | `duplicated()`、`drop_duplicates()` |
| 型別 | `pd.to_numeric(errors="coerce")`、`pd.to_datetime()` |
| 編碼 | `map({...})`、`replace()`、`.str.replace()` |
| 缺值 | `isna().sum()`、`fillna()`、`dropna()` |
| 合併 | `merge(on=, how=, validate=)` |
| 分組 | `pd.cut()`、`groupby().agg()`、`pd.crosstab()` |
| 存檔 | `to_csv(index=False)` |